[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/02_Architecture.ipynb)

# Module 2: Model Architecture Deep Dive

**Learning Objectives:**
- Understand modern transformer architecture for causal language modeling
- Learn RMSNorm, RoPE, Grouped Query Attention (GQA), and MoE
- Inspect the MiniMind model structure and parameter counts
- Trace a forward pass through the model

In [ ]:
# @title 🔧 Environment Setup
import os
if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')
!pip install -q transformers==4.57.6 torch
print("✅ Setup complete!")

## 1. Transformer Architecture Overview

MiniMind follows the **decoder-only transformer** architecture (similar to GPT/LLaMA). The key components are:

```
Input Tokens
    │
    ▼
Embedding (vocab_size=6400, hidden_size=768)
    │
    ▼
┌────────────────────────────────┐
│  MiniMindBlock (×8 layers)     │
│  ┌──────────────────────────┐ │
│  │ RMSNorm → Attention → +  │ │
│  │ RMSNorm → FFN → +       │ │
│  └──────────────────────────┘ │
└────────────────────────────────┘
    │
    ▼
RMSNorm → LM Head (weight-tied with Embedding)
    │
    ▼
Logits (vocab_size=6400)
```

Key design choices in MiniMind:
- **RMSNorm** instead of LayerNorm (simpler, faster)
- **RoPE** for positional encoding (rotary position embeddings)
- **GQA** (Grouped Query Attention) with 8 query heads and 4 KV heads
- **SiLU-gated FFN** (like LLaMA/Mistral)
- **Optional MoE** (Mixture of Experts) for the feed-forward layers
- **Weight tying** between embedding and LM head

In [ ]:
import torch
import sys
sys.path.insert(0, '.')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

# Create model with default config
config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
model = MiniMindForCausalLM(config)

# Print architecture
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## 2. RMSNorm (Root Mean Square Normalization)

MiniMind uses **RMSNorm** instead of the traditional LayerNorm. RMSNorm is simpler and faster because it skips the mean-centering step.

**LayerNorm:**
$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

**RMSNorm:**
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \cdot \gamma$$

Key differences:
- No mean subtraction (no re-centering)
- No bias term $\beta$
- Only a learnable scale parameter $\gamma$ (called `weight`)
- Computationally cheaper — fewer operations per normalization

In [ ]:
from model.model_minimind import RMSNorm
import torch

norm = RMSNorm(768)
x = torch.randn(1, 10, 768)
out = norm(x)
print(f"Input shape: {x.shape}, Output shape: {out.shape}")
print(f"Input mean: {x.mean():.4f}, Output mean: {out.mean():.4f}")
print(f"Input std: {x.std():.4f}, Output std: {out.std():.4f}")

## 3. Rotary Position Embedding (RoPE)

RoPE encodes positional information by **rotating** query and key vectors in the attention mechanism. Instead of adding position embeddings to the input, RoPE applies a rotation matrix based on position.

For a pair of dimensions $(x_i, x_{i+1})$ at position $m$:

$$\begin{pmatrix} x_i' \\ x_{i+1}' \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_i \\ x_{i+1} \end{pmatrix}$$

where $\theta_i = \text{base}^{-2i/d}$ and `base` (rope_theta) = 1,000,000 in MiniMind.

**Why RoPE?**
- Relative position information is encoded in the dot product of rotated Q and K vectors
- Works naturally with linear attention and FlashAttention
- Supports length extrapolation (especially with YaRN scaling)

MiniMind also implements **YaRN** (Yet another RoPE extensioN) for extending context length beyond training.

In [ ]:
import matplotlib.pyplot as plt
from model.model_minimind import precompute_freqs_cis
import torch

freqs_cos, freqs_sin = precompute_freqs_cis(dim=96, end=512)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(freqs_cos[:128, :48].numpy(), aspect='auto', cmap='RdBu')
axes[0].set_title('RoPE Cosine Frequencies'); axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('Position')
axes[1].imshow(freqs_sin[:128, :48].numpy(), aspect='auto', cmap='RdBu')
axes[1].set_title('RoPE Sine Frequencies'); axes[1].set_xlabel('Dimension'); axes[1].set_ylabel('Position')
plt.tight_layout(); plt.show()

## 4. Grouped Query Attention (GQA)

MiniMind uses **Grouped Query Attention (GQA)** — a middle ground between Multi-Head Attention (MHA) and Multi-Query Attention (MQA).

| Method | Query Heads | KV Heads | KV Cache Size |
|--------|------------|----------|---------------|
| MHA    | 8          | 8        | 1× (baseline) |
| GQA    | 8          | 4        | 0.5×          |
| MQA    | 8          | 1        | 0.125×        |

In MiniMind's GQA configuration:
- **8 query heads** each with dimension 96 (head_dim)
- **4 key/value heads** shared across query head groups
- Each KV head serves 2 query heads (ratio = 8/4 = 2)
- The `repeat_kv` function duplicates KV heads to match query head count

MiniMind also applies **QK-norm** (RMSNorm on Q and K after projection) for training stability.

GQA reduces the KV cache memory by 50% compared to standard MHA while maintaining most of the model quality.

In [ ]:
attn = model.model.layers[0].self_attn
print(f"Query proj: {attn.q_proj.weight.shape} \u2192 {config.num_attention_heads} heads")
print(f"Key proj:   {attn.k_proj.weight.shape} \u2192 {config.num_key_value_heads} KV heads")
print(f"Value proj: {attn.v_proj.weight.shape} \u2192 {config.num_key_value_heads} KV heads")
print(f"Output proj: {attn.o_proj.weight.shape}")
print(f"\nGQA ratio: {config.num_attention_heads // config.num_key_value_heads} query heads per KV head")
print(f"Head dimension: {config.head_dim}")

## 5. SiLU-Gated Feed-Forward Network

MiniMind uses a **SiLU-gated FFN** (also called SwiGLU variant), which is the standard in modern LLMs like LLaMA and Mistral.

**Standard FFN:**
$$\text{FFN}(x) = W_2 \cdot \text{ReLU}(W_1 x)$$

**SiLU-Gated FFN (MiniMind):**
$$\text{FFN}(x) = W_{\text{down}} \cdot (\text{SiLU}(W_{\text{gate}} x) \odot W_{\text{up}} x)$$

where SiLU (Sigmoid Linear Unit) is:
$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

The intermediate size is computed as $\lceil 768 \times \pi / 64 \rceil \times 64$, which gives a clean multiple of 64 for hardware efficiency.

Three separate linear projections:
- **gate**: Projects to intermediate size, then applies SiLU activation
- **up**: Projects to intermediate size (no activation)
- **down**: Projects back from intermediate size to hidden size

The gating mechanism allows the network to learn which features to pass through, improving gradient flow.

In [ ]:
import math

ffn = model.model.layers[0].feed_forward
print("=== SiLU-Gated FFN Structure ===")
print(f"Gate proj: {ffn.gate.weight.shape}  (hidden \u2192 intermediate)")
print(f"Up proj:   {ffn.up.weight.shape}  (hidden \u2192 intermediate)")
print(f"Down proj: {ffn.down.weight.shape}  (intermediate \u2192 hidden)")

# Verify intermediate size calculation
expected_intermediate = math.ceil(768 * math.pi / 64) * 64
print(f"\nExpected intermediate size: ceil(768 * \u03c0 / 64) * 64 = {expected_intermediate}")
print(f"Actual intermediate size: {config.intermediate_size}")

ffn_params = sum(p.numel() for p in ffn.parameters())
print(f"\nFFN parameters: {ffn_params:,} ({ffn_params/1e6:.2f}M)")

## 6. Mixture of Experts (MoE)

MiniMind optionally supports **Mixture of Experts (MoE)**, where the FFN layer is replaced by multiple expert FFN networks with a learned router.

**How MoE works in MiniMind:**
1. A **router gate** (linear layer) scores each expert for each token
2. **Top-k experts** are selected per token (e.g., top-1 out of 4 experts)
3. Selected expert outputs are combined weighted by their softmax scores
4. An **auxiliary loss** encourages balanced expert utilization

```
Input x
    │
    ├────────────────────────────┐
    │                            │
    ▼                            ▼
Router Gate               Expert Pool
(hidden → N)          [FFN_0, FFN_1, FFN_2, FFN_3]
    │                            │
    ▼                            ▼
Top-k selection    →   Weighted combination
    │
    ▼
Output y
```

**Benefits of MoE:**
- Increases model capacity without proportionally increasing compute
- Each token only activates a subset of parameters
- Total params grow, but active params per token stay manageable

In [ ]:
moe_config = MiniMindConfig(
    hidden_size=768,
    num_hidden_layers=8,
    use_moe=True,
    num_experts=4,
    num_experts_per_tok=1
)
moe_model = MiniMindForCausalLM(moe_config)

total_dense = sum(p.numel() for p in model.parameters())
total_moe = sum(p.numel() for p in moe_model.parameters())

print(f"Dense model: {total_dense/1e6:.2f}M params")
print(f"MoE model:   {total_moe/1e6:.2f}M total params")
print(f"Param increase: {(total_moe - total_dense)/1e6:.2f}M ({(total_moe/total_dense - 1)*100:.1f}% more)")

# Inspect MoE layer structure
moe_layer = moe_model.model.layers[0].feed_forward
print(f"\n=== MoE Layer Structure ===")
print(f"Router gate shape: {moe_layer.gate.weight.shape}")
print(f"Number of experts: {len(moe_layer.experts)}")
print(f"Experts per token: {moe_config.num_experts_per_tok}")

## 7. Full Forward Pass

Let's trace a complete forward pass through MiniMind:

1. **Tokenization**: Text → token IDs (vocab_size=6400)
2. **Embedding**: Token IDs → hidden states `[batch, seq_len, 768]`
3. **For each of 8 transformer blocks:**
   - RMSNorm → Grouped Query Attention (with RoPE) → + residual
   - RMSNorm → SiLU-gated FFN (or MoE) → + residual
4. **Final RMSNorm**: Normalize the output
5. **LM Head**: Project to vocabulary logits `[batch, seq_len, 6400]`
6. **Sampling**: Apply temperature, top-p, top-k, repetition penalty

The model uses **causal masking** so each token can only attend to previous tokens (autoregressive generation).

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('./model')

text = "Hello world"
inputs = tokenizer(text, return_tensors="pt")
print(f"Input IDs shape: {inputs['input_ids'].shape}")
print(f"Input IDs: {inputs['input_ids']}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")

with torch.no_grad():
    outputs = model(**inputs)

print(f"\nLogits shape: {outputs.logits.shape}")
print(f"Vocab size: {config.vocab_size}")

# Show predicted next tokens (model is untrained, so predictions are random)
probs = torch.softmax(outputs.logits[0, -1], dim=-1)
top5 = torch.topk(probs, 5)
print("\nTop 5 predicted next tokens (untrained model):")
for i, (prob, idx) in enumerate(zip(top5.values, top5.indices)):
    print(f"  Top {i+1}: '{tokenizer.decode([idx])}' (p={prob:.4f})")

## 8. Weight Tying

MiniMind uses **weight tying** — the embedding layer and the final LM head share the same weight matrix.

$$W_{\text{embed\_tokens}} = W_{\text{lm\_head}}$$

**Why weight tying?**
- Reduces total parameter count (the embedding matrix is large: `vocab_size × hidden_size = 6400 × 768 ≈ 4.9M` params)
- Enforces consistency: tokens with similar embeddings produce similar output logits
- Standard practice in modern LLMs (GPT-2, LLaMA, etc.)

Without weight tying, the model would have ~4.9M additional parameters.

In [ ]:
# Verify weight tying: embedding and LM head share the same tensor
embed_weight = model.model.embed_tokens.weight
lm_head_weight = model.lm_head.weight

print(f"Embedding weight shape: {embed_weight.shape}")
print(f"LM head weight shape:   {lm_head_weight.shape}")
print(f"\nSame tensor (data_ptr match): {embed_weight.data_ptr() == lm_head_weight.data_ptr()}")
print(f"Weights are equal: {torch.equal(embed_weight, lm_head_weight)}")

# Count parameters with and without tying
total_params = sum(p.numel() for p in model.parameters())
embed_params = embed_weight.numel()
print(f"\nTotal parameters (with tying): {total_params:,}")
print(f"Embedding/LM head size: {embed_params:,} ({embed_params/1e6:.2f}M)")
print(f"Without tying would be: {(total_params + embed_params):,} ({(total_params + embed_params)/1e6:.2f}M)")

## 📝 Exercises

1. **Modify the config**: Try changing `hidden_size` to 512 or 1024. How does the parameter count change? What about `num_hidden_layers`?

2. **Explore GQA ratios**: Change `num_key_value_heads` to 1 (MQA), 2, 4 (default), or 8 (MHA). How do the attention weight shapes change?

3. **MoE experiments**: Try different `num_experts` (2, 4, 8) and `num_experts_per_tok` (1, 2) values. What's the tradeoff between total params and active params?

4. **RoPE frequencies**: Visualize how changing `rope_theta` (try 10000 vs 1000000) affects the frequency patterns.

5. **Layer-by-layer analysis**: Write code to count parameters in each component (attention, FFN, norms, embeddings) as a percentage of total.

## ⮕️ Next Steps

In **Module 3**, we'll dive into the training process:
- Dataset preparation and tokenization
- Pre-training with causal language modeling
- Loss functions and optimization
- Training monitoring and evaluation